In [2]:
from faker import Faker
import pandas as pd
import random
from datetime import date, timedelta

In [3]:
fake = Faker("en_US")

Faker.seed(42)
random.seed(42)

In [23]:
# Número de registros
NUM_CUSTOMERS = 500
NUM_CATEGORIES = 6
NUM_PRODUCTS = 70
NUM_ORDERS = 2000
TARGET_ORDER_ITEMS = 4500

**Categories**

In [4]:
CATEGORIES = [
    {
        "name": "Smartphones",
        "description": "Smartphones and mobile devices"
    },
    {
        "name": "Laptops",
        "description": "Laptops and portable computers"
    },
    {
        "name": "Tablets",
        "description": "Tablets and portable touchscreen devices"
    },
    {
        "name": "Audio",
        "description": "Headphones, speakers and audio equipment"
    },
    {
        "name": "Gaming",
        "description": "Gaming consoles, accessories and equipment"
    },
    {
        "name": "Accessories",
        "description": "Technology accessories and peripherals"
    }
]

In [11]:
def generate_categories():
    categories = []

    for category_id, category in enumerate(CATEGORIES, start=1):
        categories.append({
            "category_id": category_id,
            "name": category["name"],
            "description": category["description"]
        })

    return pd.DataFrame(categories)

categories_df = generate_categories()

print(categories_df)

   category_id         name                                 description
0            1  Smartphones              Smartphones and mobile devices
1            2      Laptops              Laptops and portable computers
2            3      Tablets    Tablets and portable touchscreen devices
3            4        Audio    Headphones, speakers and audio equipment
4            5       Gaming  Gaming consoles, accessories and equipment
5            6  Accessories      Technology accessories and peripherals


**Products**

In [8]:
PRODUCTS_BY_CATEGORY = {
    "Smartphones": [
        "iPhone 15",
        "iPhone 15 Pro",
        "iPhone 15 Pro Max",
        "iPhone 16",
        "iPhone 16 Pro",
        "Samsung Galaxy S24",
        "Samsung Galaxy S24 Ultra",
        "Samsung Galaxy S25",
        "Samsung Galaxy A55",
        "Google Pixel 8",
        "Google Pixel 8 Pro",
        "Google Pixel 9",
        "Xiaomi 14",
        "Xiaomi 14 Pro",
        "OnePlus 12"
    ],

    "Laptops": [
        "MacBook Air M2",
        "MacBook Air M3",
        "MacBook Pro 14 M3",
        "MacBook Pro 16 M3",
        "Dell XPS 13",
        "Dell XPS 15",
        "Dell Inspiron 15",
        "Lenovo ThinkPad E14",
        "Lenovo ThinkPad X1 Carbon",
        "Lenovo IdeaPad 5",
        "HP Pavilion 15",
        "ASUS Zenbook 14"
    ],

    "Tablets": [
        "iPad 10th Gen",
        "iPad Air",
        "iPad Air M2",
        "iPad Pro 11",
        "iPad Pro 13",
        "Samsung Galaxy Tab S9",
        "Samsung Galaxy Tab S9 Ultra",
        "Samsung Galaxy Tab A9",
        "Lenovo Tab P12",
        "Xiaomi Pad 6"
    ],

    "Audio": [
        "AirPods 2nd Gen",
        "AirPods 3rd Gen",
        "AirPods Pro 2",
        "Sony WH-1000XM5",
        "Sony WH-CH720N",
        "Bose QuietComfort",
        "Bose QuietComfort Ultra",
        "JBL Live 660NC",
        "JBL Flip 6",
        "Sonos Era 100"
    ],

    "Gaming": [
        "PlayStation 5",
        "PlayStation 5 Slim",
        "Xbox Series X",
        "Xbox Series S",
        "Nintendo Switch OLED",
        "Nintendo Switch Lite",
        "Steam Deck OLED",
        "ASUS ROG Ally",
        "Logitech G Pro X Superlight",
        "Razer BlackShark V2",
        "Elgato Stream Deck",
        "Corsair K70 RGB"
    ],

    "Accessories": [
        "Apple MagSafe Charger",
        "USB-C Fast Charger",
        "Anker 65W Charger",
        "USB-C Hub 7-in-1",
        "Lightning Cable",
        "USB-C Cable 2m",
        "Samsung 25W Charger",
        "SanDisk 1TB SSD",
        "Samsung 1TB SSD",
        "Logitech MX Master 3S",
        "Apple Magic Keyboard"
    ]
}

PRICE_RANGES = {
    "Smartphones": (150, 1500),
    "Laptops": (400, 2500),
    "Tablets": (150, 1500),
    "Audio": (30, 500),
    "Gaming": (40, 1000),
    "Accessories": (10, 300)
}

In [12]:
def generate_products(categories_df):
    products = []
    product_id = 1

    for category in PRODUCTS_BY_CATEGORY:
        
        category_id = categories_df.loc[
            categories_df["name"] == category,
            "category_id"
        ].iloc[0]

        min_price, max_price = PRICE_RANGES[category]

        for product_name in PRODUCTS_BY_CATEGORY[category]:

            sale_price = round(
                random.uniform(min_price, max_price),
                2
            )

            cost_price = round(
                sale_price * random.uniform(0.60, 0.85),
                2
            )

            stock = random.randint(0, 150)

            is_active = random.random() < 0.90

            products.append({
                "product_id": product_id,
                "category_id": category_id,
                "name": product_name,
                "description": fake.text(max_nb_chars=150),
                "sale_price": sale_price,
                "cost_price": cost_price,
                "stock": stock,
                "is_active": is_active
            })

            product_id += 1

    return pd.DataFrame(products)

**Customers**

In [15]:
ACQUISITION_CHANNELS = [
    "Organic Search",
    "Paid Search",
    "Social Media",
    "Email",
    "Referral",
    "Direct",
    "Affiliate"
]

In [20]:
def generate_customers():
    customers = []

    start_date = date(2022, 1, 1)
    end_date = date(2025, 12, 31)

    for customer_id in range(1, NUM_CUSTOMERS + 1):

        registration_date = fake.date_between(
            start_date=start_date,
            end_date=end_date
        )

        customers.append({
            "customer_id": customer_id,
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": fake.unique.email(),
            "phone": fake.unique.phone_number(),
            "country": fake.country(),
            "city": fake.city(),
            "acquisition_channel": random.choice(ACQUISITION_CHANNELS),
            "registration_date": registration_date
        })

    return pd.DataFrame(customers)

**Orders**

In [33]:
ORDER_STATUSES = [
    "pending",
    "confirmed",
    "shipped",
    "delivered",
    "cancelled",
    "returned"
]

ORDER_STATUS_WEIGHTS = [
    0.08,  # pending
    0.12,  # confirmed
    0.12,  # shipped
    0.55,  # delivered
    0.08,  # cancelled
    0.05   # returned
]

In [30]:
def generate_orders(customers_df):
    orders = []

    for order_id in range(1, NUM_ORDERS + 1):

        # Seleccionar un cliente existente
        customer = customers_df.sample(
            n=1,
            random_state=order_id
        ).iloc[0]

        customer_id = customer["customer_id"]
        registration_date = customer["registration_date"]

        order_date = fake.date_between(
            start_date=registration_date,
            end_date=date(2025, 12, 31)
        )

        # Elegir estado según probabilidades
        order_status = random.choices(
            ORDER_STATUSES,
            weights=ORDER_STATUS_WEIGHTS,
            k=1
        )[0]

        # Datos de envío 
        shipping_address = fake.street_address()
        shipping_city = fake.city()
        shipping_country = fake.country()

        # Inicialmente no hay fechas de envío ni entrega
        shipped_date = None
        delivered_date = None

        # Los pedidos enviados, entregados y devueltos
        # tienen fecha de envío
        if order_status in ["shipped", "delivered", "returned"]:

            shipped_date = order_date + timedelta(
                days=random.randint(1, 5)
            )

        # Los pedidos entregados y devueltos
        # tienen fecha de entrega
        if order_status in ["delivered", "returned"]:

            delivered_date = shipped_date + timedelta(
                days=random.randint(1, 7)
            )

        orders.append({
            "order_id": order_id,
            "customer_id": customer_id,
            "order_status": order_status,
            "shipping_address": shipping_address,
            "shipping_city": shipping_city,
            "shipping_country": shipping_country,
            "order_date": order_date,
            "shipped_date": shipped_date,
            "delivered_date": delivered_date
        })

    return pd.DataFrame(orders)

**Order Items**

In [35]:
def generate_order_items(orders_df, products_df):
    order_items = []

    order_item_id = 1

    # Número de líneas que queremos generar inicialmente
    items_per_order = []

    for _ in range(NUM_ORDERS):
        items_per_order.append(
            random.choices(
                [1, 2, 3, 4, 5],
                weights=[0.10, 0.40, 0.35, 0.10, 0.05],
                k=1
            )[0]
        )

    # Ajustar hasta alcanzar exactamente 4.500 líneas
    difference = TARGET_ORDER_ITEMS - sum(items_per_order)

    while difference != 0:

        if difference > 0:
            index = random.randrange(NUM_ORDERS)

            if items_per_order[index] < 5:
                items_per_order[index] += 1
                difference -= 1

        else:
            index = random.randrange(NUM_ORDERS)

            if items_per_order[index] > 1:
                items_per_order[index] -= 1
                difference += 1

    # Generar las líneas de pedido
    for _, order in orders_df.iterrows():

        order_id = order["order_id"]

        number_of_items = items_per_order[order_id - 1]

        # Elegir productos diferentes dentro del mismo pedido
        selected_products = random.sample(
            list(products_df["product_id"]),
            number_of_items
        )

        for product_id in selected_products:

            product = products_df[
                products_df["product_id"] == product_id
            ].iloc[0]

            quantity = random.randint(1, 3)

            unit_price = product["sale_price"]

            discount = random.choices(
                [0.00, 0.05, 0.10, 0.15, 0.20],
                weights=[0.45, 0.20, 0.20, 0.10, 0.05],
                k=1
            )[0]

            order_items.append({
                "order_item_id": order_item_id,
                "order_id": order_id,
                "product_id": product_id,
                "quantity": quantity,
                "unit_price": unit_price,
                "discount": discount
            })

            order_item_id += 1

    return pd.DataFrame(order_items)

**Payments**

In [38]:
PAYMENT_METHODS = [
    "credit_card",
    "debit_card",
    "paypal",
    "bank_transfer",
    "apple_pay",
    "google_pay"
]

PAYMENT_STATUSES = [
    "pending",
    "completed",
    "failed",
    "refunded"
]

In [40]:
def calculate_order_totals(order_items_df):
    order_items = order_items_df.copy()

    order_items["line_total"] = (
        order_items["quantity"]
        * order_items["unit_price"]
        * (1 - order_items["discount"])
    )

    order_totals = (
        order_items
        .groupby("order_id")["line_total"]
        .sum()
        .round(2)
        .reset_index(name="order_total")
    )

    return order_totals



def generate_payments(orders_df, order_items_df):
    payments = []

    payment_id = 1

    # Calcular el importe total de cada pedido
    order_totals = calculate_order_totals(order_items_df)

    # Unir el total al pedido
    orders_with_totals = orders_df.merge(
        order_totals,
        on="order_id"
    )

    for _, order in orders_with_totals.iterrows():

        order_id = order["order_id"]
        order_status = order["order_status"]
        order_total = order["order_total"]
        order_date = order["order_date"]

        payment_method = random.choice(PAYMENT_METHODS)

        # Por defecto
        payment_status = "completed"

        if order_status == "pending":
            payment_status = "pending"

        elif order_status == "cancelled":
            payment_status = "failed"

        elif order_status == "returned":
            payment_status = "refunded"

        # Fecha del pago
        payment_date = order_date + timedelta(
            days=random.randint(0, 2)
        )

        # Algunos pedidos tendrán varios pagos
        number_of_payments = random.choices(
            [1, 2],
            weights=[0.75, 0.25],
            k=1
        )[0]

        if number_of_payments == 1:

            payments.append({
                "payment_id": payment_id,
                "order_id": order_id,
                "payment_method": payment_method,
                "payment_status": payment_status,
                "amount": round(order_total, 2),
                "payment_date": payment_date
            })

            payment_id += 1

        else:

            # Dividir el importe total entre dos pagos
            first_amount = round(
                order_total * random.uniform(0.30, 0.70),
                2
            )

            second_amount = round(
                order_total - first_amount,
                2
            )

            payments.append({
                "payment_id": payment_id,
                "order_id": order_id,
                "payment_method": payment_method,
                "payment_status": payment_status,
                "amount": first_amount,
                "payment_date": payment_date
            })

            payment_id += 1

            # El segundo pago puede utilizar otro método
            second_payment_method = random.choice(
                PAYMENT_METHODS
            )

            second_payment_date = payment_date + timedelta(
                days=random.randint(0, 2)
            )

            payments.append({
                "payment_id": payment_id,
                "order_id": order_id,
                "payment_method": second_payment_method,
                "payment_status": payment_status,
                "amount": second_amount,
                "payment_date": second_payment_date
            })

            payment_id += 1

    return pd.DataFrame(payments)

**Reviews**

In [44]:
RATING_VALUES = [1, 2, 3, 4, 5]

RATING_WEIGHTS = [
    0.05,  # 1 estrella
    0.08,  # 2 estrellas
    0.17,  # 3 estrellas
    0.30,  # 4 estrellas
    0.40   # 5 estrellas
]

In [45]:
def generate_reviews(orders_df, order_items_df):
    reviews = []

    # Solo podemos valorar productos de pedidos entregados
    delivered_orders = orders_df[
        orders_df["order_status"] == "delivered"
    ][
        ["order_id", "delivered_date"]
    ]

    # Obtener las líneas correspondientes a pedidos entregados
    delivered_items = order_items_df.merge(
        delivered_orders,
        on="order_id"
    )

    # Queremos aproximadamente el 35% de las líneas entregadas
    number_of_reviews = round(
        len(delivered_items) * 0.35
    )

    # Seleccionar aleatoriamente las líneas que recibirán review
    selected_items = delivered_items.sample(
        n=number_of_reviews,
        random_state=42
    )

    review_id = 1

    for _, item in selected_items.iterrows():

        order_item_id = item["order_item_id"]
        delivered_date = item["delivered_date"]

        # Generar valoración de 1 a 5
        rating = random.choices(
            RATING_VALUES,
            weights=RATING_WEIGHTS,
            k=1
        )[0]

        # Generar comentario con Faker
        comment = fake.sentence(
            nb_words=random.randint(6, 15)
        )

        # La review se publica después de recibir el producto
        review_date = delivered_date + timedelta(
            days=random.randint(1, 14)
        )

        reviews.append({
            "review_id": review_id,
            "order_item_id": order_item_id,
            "rating": rating,
            "comment": comment,
            "review_date": review_date
        })

        review_id += 1

    return pd.DataFrame(reviews)

**COMPROBACIONES**

In [47]:
#Categories
categories_df = generate_categories()

#Products
products_df = generate_products(categories_df)

#Customers
customers_df = generate_customers()

print(categories_df)

print(products_df)
print(f"\nNúmero de productos: {len(products_df)}")

print(customers_df)
print(f"\nNúmero de clientes: {len(customers_df)}")

print("\nPrimeros 5 clientes:")
print(customers_df.head())

print("\nClientes por país:")
print(customers_df["country"].value_counts().head(10))

#Orders
orders_df = generate_orders(customers_df)

print(orders_df)

print(f"\nNúmero de pedidos: {len(orders_df)}")

print("\nPedidos por estado:")
print(orders_df["order_status"].value_counts())

#Order Items
order_items_df = generate_order_items(
    orders_df,
    products_df
)

print(order_items_df)

print(
    f"\nNúmero de líneas de pedido: "
    f"{len(order_items_df)}"
)


#Payments´
payments_df = generate_payments(
    orders_df,
    order_items_df
)

print(payments_df)

print(
    f"\nNúmero de pagos: "
    f"{len(payments_df)}"
)

orders_without_payment = (
    set(orders_df["order_id"])
    - set(payments_df["order_id"])
)

print(
    f"\nPedidos sin ningún pago: "
    f"{len(orders_without_payment)}"
)

print("--------------------------------------------")
payment_totals = (
    payments_df
    .groupby("order_id")["amount"]
    .sum()
    .round(2)
    .reset_index(name="paid_total")
)

order_totals = calculate_order_totals(
    order_items_df
)

payment_check = order_totals.merge(
    payment_totals,
    on="order_id"
)

payment_check["difference"] = (
    payment_check["order_total"]
    - payment_check["paid_total"]
).round(2)

print(
    "\nPedidos cuyo total no coincide con sus pagos:"
)

print(
    (payment_check["difference"] != 0).sum()
)

#Reviews

reviews_df = generate_reviews(
    orders_df,
    order_items_df
)

print(reviews_df)

print(
    f"\nNúmero de reviews: "
    f"{len(reviews_df)}"
)

print("\nDistribución de ratings:")
print(
    reviews_df["rating"].value_counts().sort_index()
)

   category_id         name                                 description
0            1  Smartphones              Smartphones and mobile devices
1            2      Laptops              Laptops and portable computers
2            3      Tablets    Tablets and portable touchscreen devices
3            4        Audio    Headphones, speakers and audio equipment
4            5       Gaming  Gaming consoles, accessories and equipment
5            6  Accessories      Technology accessories and peripherals
    product_id  category_id                   name  \
0            1            1              iPhone 15   
1            2            1          iPhone 15 Pro   
2            3            1      iPhone 15 Pro Max   
3            4            1              iPhone 16   
4            5            1          iPhone 16 Pro   
..         ...          ...                    ...   
65          66            6    Samsung 25W Charger   
66          67            6        SanDisk 1TB SSD   
67        

**VALIDACIÓN FINAL**

In [48]:
categories_df = generate_categories()

products_df = generate_products(categories_df)

customers_df = generate_customers()

orders_df = generate_orders(customers_df)

order_items_df = generate_order_items(
    orders_df,
    products_df
)

payments_df = generate_payments(
    orders_df,
    order_items_df
)

reviews_df = generate_reviews(
    orders_df,
    order_items_df
)

In [49]:
print("customers:", len(customers_df))
print("categories:", len(categories_df))
print("products:", len(products_df))
print("orders:", len(orders_df))
print("order_items:", len(order_items_df))
print("payments:", len(payments_df))
print("reviews:", len(reviews_df))

customers: 500
categories: 6
products: 70
orders: 2000
order_items: 4500
payments: 2517
reviews: 842


In [50]:
print("CUSTOMERS")
print(customers_df.shape)
print(customers_df.columns.tolist())

print("\nCATEGORIES")
print(categories_df.shape)
print(categories_df.columns.tolist())

print("\nPRODUCTS")
print(products_df.shape)
print(products_df.columns.tolist())

print("\nORDERS")
print(orders_df.shape)
print(orders_df.columns.tolist())

print("\nORDER_ITEMS")
print(order_items_df.shape)
print(order_items_df.columns.tolist())

print("\nPAYMENTS")
print(payments_df.shape)
print(payments_df.columns.tolist())

print("\nREVIEWS")
print(reviews_df.shape)
print(reviews_df.columns.tolist())

CUSTOMERS
(500, 9)
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'country', 'city', 'acquisition_channel', 'registration_date']

CATEGORIES
(6, 3)
['category_id', 'name', 'description']

PRODUCTS
(70, 8)
['product_id', 'category_id', 'name', 'description', 'sale_price', 'cost_price', 'stock', 'is_active']

ORDERS
(2000, 9)
['order_id', 'customer_id', 'order_status', 'shipping_address', 'shipping_city', 'shipping_country', 'order_date', 'shipped_date', 'delivered_date']

ORDER_ITEMS
(4500, 6)
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'discount']

PAYMENTS
(2517, 6)
['payment_id', 'order_id', 'payment_method', 'payment_status', 'amount', 'payment_date']

REVIEWS
(842, 5)
['review_id', 'order_item_id', 'rating', 'comment', 'review_date']


In [51]:
orders_df["order_status"].value_counts()

order_status
delivered    1059
confirmed     263
shipped       257
pending       168
cancelled     149
returned      104
Name: count, dtype: int64

In [52]:
products_df["category_id"].value_counts().sort_index()

category_id
1    15
2    12
3    10
4    10
5    12
6    11
Name: count, dtype: int64

In [53]:
payments_df["payment_method"].value_counts()

payment_method
bank_transfer    431
paypal           430
google_pay       424
debit_card       416
credit_card      413
apple_pay        403
Name: count, dtype: int64

In [54]:
reviews_df["rating"].value_counts().sort_index()

rating
1     45
2     72
3    153
4    229
5    343
Name: count, dtype: int64

In [56]:
review_check = reviews_df.merge(
    order_items_df[["order_item_id", "order_id"]],
    on="order_item_id"
).merge(
    orders_df[["order_id", "order_status"]],
    on="order_id"
)

review_check["order_status"].value_counts()

order_status
delivered    842
Name: count, dtype: int64

In [57]:
order_items_df["discount"].value_counts().sort_index()

discount
0.00    1979
0.05     888
0.10     907
0.15     471
0.20     255
Name: count, dtype: int64

In [59]:
categories_df = generate_categories()
products_df = generate_products(categories_df)
customers_df = generate_customers()
orders_df = generate_orders(customers_df)
order_items_df = generate_order_items(orders_df, products_df)
payments_df = generate_payments(orders_df, order_items_df)
reviews_df = generate_reviews(orders_df, order_items_df)

In [63]:
customers_df.to_csv("../data/customers.csv", index=False)

categories_df.to_csv("../data/categories.csv", index=False)

products_df.to_csv("../data/products.csv", index=False)

orders_df.to_csv("../data/orders.csv", index=False)

order_items_df.to_csv("../data/order_items.csv", index=False)

payments_df.to_csv("../data/payments.csv", index=False)

reviews_df.to_csv("../data/reviews.csv", index=False)